### Step 1 — Load Raw Data

Load all four input files into separate raw dataframes for inspection.

In [1]:
import pandas as pd

# ── Load Raw Data ──────────────────────────────────────────────────────────────
df_raw_consumption = pd.read_excel(r"C:\Users\Amey\Desktop\Amey\Python\Consumption_data.xlsx")
df_raw_equip_no    = pd.read_excel(r"C:\Users\Amey\Desktop\Amey\Python\Equipment No and OBR Data.xlsx")
df_raw_temp        = pd.read_csv(r"C:\Users\Amey\Desktop\Amey\Python\clean_temperature_data.csv")
df_raw_region      = pd.read_csv(r"C:\Users\Amey\Desktop\Amey\Python\cities.csv")

# ── Quick Verification ─────────────────────────────────────────────────────────
for name, df in [("df_raw_consumption", df_raw_consumption),
                 ("df_raw_equip_no",    df_raw_equip_no),
                 ("df_raw_temp",        df_raw_temp),
                 ("df_raw_region",      df_raw_region)]:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {df.columns.tolist()}\n")

df_raw_consumption: 64,895 rows × 18 columns
  Columns: ['Material', 'Material Description', 'MvT', 'SLoc', 'User name', 'Sales Ord.', '   Quantity', 'EUn', 'Pstng Date', 'Reference', 'PO', 'WBS Elem.', 'Entry Date', 'Doc. Date', 'Order', 'Vendor', 'Plnt', 'Text']

df_raw_equip_no: 20,283 rows × 18 columns
  Columns: ['CN', 'NC #', 'Order', 'Creation date', 'Status', 'Requested deliv.date', 'Handover to customer', 'Short Text', 'Equipment', 'Equi. long address', '    Net value', 'Sell hours', 'Technician name', 'Technician 2 No', 'NC Aging', 'Risk Flag', 'NC Rej. by', 'NC Rej. on']

df_raw_temp: 202,797 rows × 11 columns
  Columns: ['SLOC', 'Location', 'Date', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Month', 'Year', 'Season', 'Delta_T']

df_raw_region: 87 rows × 4 columns
  Columns: ['SLOC', 'City', 'Location', 'Region']



### Step 2 — Remove Unnecessary Columns

Drop all columns not required for analysis from the Consumption, Equipment/OBR, and Region dataframes.  
The temperature dataframe already contains only the required columns so it is saved as-is.

In [2]:
# ── df_raw_consumption: Drop unnecessary columns ───────────────────────────────
cols_to_drop_consumption = [
    'Material Description', 'MvT', 'User name', 'EUn', 'Reference',
    'PO', 'WBS Elem.', 'Entry Date', 'Doc. Date', 'Order', 'Vendor', 'Plnt', 'Text'
]
df_req_col_consumption = df_raw_consumption.drop(columns=cols_to_drop_consumption)

# ── df_raw_equip_no: Drop unnecessary columns ──────────────────────────────────
cols_to_drop_equip_no = [
    'CN', 'NC #', 'Creation date', 'Status', 'Requested deliv.date',
    'Handover to customer', 'Short Text', 'Equi. long address', '    Net value',
    'Sell hours', 'Technician 2 No', 'NC Aging', 'Risk Flag', 'NC Rej. by', 'NC Rej. on'
]
df_req_col_equip_no = df_raw_equip_no.drop(columns=cols_to_drop_equip_no)

# ── df_raw_temp: No columns to drop, save as-is ───────────────────────────────
df_req_col_temp = df_raw_temp.copy()

# ── df_raw_region: Drop City and Location columns ─────────────────────────────
df_req_region = df_raw_region.drop(columns=['City', 'Location'])

# ── Verification ───────────────────────────────────────────────────────────────
for name, df in [("df_req_col_consumption", df_req_col_consumption),
                 ("df_req_col_equip_no",    df_req_col_equip_no),
                 ("df_req_col_temp",        df_req_col_temp),
                 ("df_req_region",          df_req_region)]:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {df.columns.tolist()}\n")

df_req_col_consumption: 64,895 rows × 5 columns
  Columns: ['Material', 'SLoc', 'Sales Ord.', '   Quantity', 'Pstng Date']

df_req_col_equip_no: 20,283 rows × 3 columns
  Columns: ['Order', 'Equipment', 'Technician name']

df_req_col_temp: 202,797 rows × 11 columns
  Columns: ['SLOC', 'Location', 'Date', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Month', 'Year', 'Season', 'Delta_T']

df_req_region: 87 rows × 2 columns
  Columns: ['SLOC', 'Region']



### Step 3 — Standardise Column Names and Data Types

Strip whitespace from all column names first, then rename columns to match the agreed schema  
and enforce the correct data types across all four dataframes.  
Note: Equipment Number is kept as String because some values contain hyphens (e.g. 10687716-1).

In [3]:
# ── Strip whitespace from ALL column names first (handles hidden spaces) ───────
df_req_col_consumption.columns = df_req_col_consumption.columns.str.strip()
df_req_col_equip_no.columns    = df_req_col_equip_no.columns.str.strip()
df_req_col_temp.columns        = df_req_col_temp.columns.str.strip()
df_req_region.columns          = df_req_region.columns.str.strip()

# ── df_req_col_consumption ─────────────────────────────────────────────────────
df_req_col_consumption = df_req_col_consumption.rename(columns={
    'Sales Ord.' : 'Order',
    'Pstng Date' : 'Date'
})

df_req_col_consumption['Material'] = df_req_col_consumption['Material'].astype(str)
df_req_col_consumption['SLoc']     = df_req_col_consumption['SLoc'].astype(str)
df_req_col_consumption['Order']    = df_req_col_consumption['Order'].astype(str)
df_req_col_consumption['Quantity'] = df_req_col_consumption['Quantity'].astype(int)
df_req_col_consumption['Date']     = pd.to_datetime(df_req_col_consumption['Date'])

# ── df_req_col_equip_no ────────────────────────────────────────────────────────
df_req_col_equip_no = df_req_col_equip_no.rename(columns={
    'Equipment'      : 'Equipment Number',
    'Technician name': 'Technician Name'
})

df_req_col_equip_no['Order']            = df_req_col_equip_no['Order'].astype(str)
df_req_col_equip_no['Equipment Number'] = df_req_col_equip_no['Equipment Number'].astype(str)
df_req_col_equip_no['Technician Name']  = df_req_col_equip_no['Technician Name'].astype(str)

# ── df_req_col_temp ────────────────────────────────────────────────────────────
df_req_col_temp = df_req_col_temp.rename(columns={
    'SLOC' : 'SLoc'
})

df_req_col_temp['SLoc']     = df_req_col_temp['SLoc'].astype(str)
df_req_col_temp['Location'] = df_req_col_temp['Location'].astype(str)
df_req_col_temp['Date']     = pd.to_datetime(df_req_col_temp['Date'])
df_req_col_temp['Tavg']     = df_req_col_temp['Tavg'].astype(float)
df_req_col_temp['Tmax']     = df_req_col_temp['Tmax'].astype(float)
df_req_col_temp['Tmin']     = df_req_col_temp['Tmin'].astype(float)
df_req_col_temp['RH']       = df_req_col_temp['RH'].astype(float)
df_req_col_temp['Month']    = df_req_col_temp['Month'].astype(int)
df_req_col_temp['Year']     = df_req_col_temp['Year'].astype(int)
df_req_col_temp['Season']   = df_req_col_temp['Season'].astype(str)
df_req_col_temp['Delta_T']  = df_req_col_temp['Delta_T'].astype(float)

# ── df_req_region ──────────────────────────────────────────────────────────────
df_req_region = df_req_region.rename(columns={
    'SLOC' : 'SLoc'
})

df_req_region['SLoc']   = df_req_region['SLoc'].astype(str)
df_req_region['Region'] = df_req_region['Region'].astype(str)

# ── Verification ───────────────────────────────────────────────────────────────
for name, df in [("df_req_col_consumption", df_req_col_consumption),
                 ("df_req_col_equip_no",    df_req_col_equip_no),
                 ("df_req_col_temp",        df_req_col_temp),
                 ("df_req_region",          df_req_region)]:
    print(f"{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print(df.dtypes)
    print()
    display(df.head(2))
    print()

df_req_col_consumption
Material               str
SLoc                   str
Order                  str
Quantity             int64
Date        datetime64[us]
dtype: object



,Material,SLoc,Order,Quantity,Date
0,111,5043,68160889,-1,2026-01-01
1,111,5043,68160889,1,2026-01-01



df_req_col_equip_no
Order               str
Equipment Number    str
Technician Name     str
dtype: object



,Order,Equipment Number,Technician Name
0,68631760,10853186,Sanjay Name
1,68526738,10667512,Saurabh Borle



df_req_col_temp
SLoc                   str
Location               str
Date        datetime64[us]
Tavg               float64
Tmax               float64
Tmin               float64
RH                 float64
Month                int64
Year                 int64
Season                 str
Delta_T            float64
dtype: object



,SLoc,Location,Date,Tavg,Tmax,Tmin,RH,Month,Year,Season,Delta_T
0,5015,Dadra and Nagar Ha,2020-01-01,18.76,26.88,12.06,69.16,1,2020,Winter,14.82
1,5015,Dadra and Nagar Ha,2020-01-02,19.76,27.71,14.27,75.41,1,2020,Winter,13.44



df_req_region
SLoc      str
Region    str
dtype: object



,SLoc,Region
0,5015,East
1,5019,West 1


### Step 4 — Create df_combined

Merge all dataframes into a single combined dataframe using left joins.  
- Equipment Number and Technician Name are mapped from df_req_col_equip_no using Order as the key.  
- Weather columns are mapped from df_req_col_temp using SLoc + Date as the key.  
- Region is mapped from df_req_region using SLoc as the key.  
All joins are left joins so every consumption row is retained even without a match.

In [4]:
# ── Merge 1: Consumption ← Equipment/OBR on Order ─────────────────────────────
df_combined = df_req_col_consumption.merge(
    df_req_col_equip_no,
    on  = 'Order',
    how = 'left'
)

# ── Merge 2: Result ← Temperature on SLoc + Date ──────────────────────────────
df_combined = df_combined.merge(
    df_req_col_temp,
    on  = ['SLoc', 'Date'],
    how = 'left'
)

# ── Merge 3: Result ← Region on SLoc ──────────────────────────────────────────
df_combined = df_combined.merge(
    df_req_region,
    on  = 'SLoc',
    how = 'left'
)

# ── Reorder columns as per agreed structure ────────────────────────────────────
df_combined = df_combined[[
    'Material', 'SLoc', 'Order', 'Quantity',
    'Equipment Number', 'Technician Name',
    'Tavg', 'Tmax', 'Tmin', 'RH', 'Delta_T',
    'Date', 'Month', 'Year', 'Season', 'Location', 'Region'
]]

# ── Verification ───────────────────────────────────────────────────────────────
print(f"Shape   : {df_combined.shape[0]:,} rows × {df_combined.shape[1]} columns")
print(f"Columns : {df_combined.columns.tolist()}")
print()
display(df_combined.head(5))

Shape   : 64,895 rows × 17 columns
Columns : ['Material', 'SLoc', 'Order', 'Quantity', 'Equipment Number', 'Technician Name', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Delta_T', 'Date', 'Month', 'Year', 'Season', 'Location', 'Region']



,Material,SLoc,Order,Quantity,Equipment Number,Technician Name,Tavg,Tmax,Tmin,RH,Delta_T,Date,Month,Year,Season,Location,Region
0,111,5043,68160889,-1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
1,111,5043,68160889,1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
2,100,5030,68165456,-1,11565095,Kamal Kant,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
3,111,5030,68165435,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
4,111,5030,68165114,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1


### Step 5 — Data Quality Checks

Check for missing and duplicate Order numbers in df_combined, and missing SLoc values  
that could not be matched to the temperature/region data.  
Results are saved to separate CSV files for review and correction.

In [5]:
# ── Step 5: Data Quality Checks ───────────────────────────────────────────────

import os
os.makedirs("Logs", exist_ok=True)

# ── Missing Orders ─────────────────────────────────────────────────────────────
df_missing_orders = df_combined[
    df_combined['Order'].isna() | (df_combined['Order'] == 'nan')
]
df_missing_orders.to_csv("Logs/missing_order.csv", index=False)
print(f"Missing Orders       : {len(df_missing_orders):,} rows → Logs/missing_order.csv")

# ── Missing SLoc ───────────────────────────────────────────────────────────────
df_missing_sloc = df_combined[
    df_combined['Tavg'].isna() | df_combined['Region'].isna()
]
df_missing_sloc.to_csv("Logs/missing_sloc.csv", index=False)
print(f"Missing SLoc         : {len(df_missing_sloc):,} rows → Logs/missing_sloc.csv")

# ── Merge Fan-out Check ────────────────────────────────────────────────────────
equip_per_order = df_req_col_equip_no.groupby('Order')['Equipment Number'].nunique()
fanout_orders   = equip_per_order[equip_per_order > 1]
if len(fanout_orders) > 0:
    fanout_orders.reset_index().to_csv("Logs/fanout_orders.csv", index=False)
    print(f"Merge Fan-out Orders : {len(fanout_orders):,} orders → Logs/fanout_orders.csv ⚠️")
else:
    print(f"Merge Fan-out Orders : 0 (clean)")

# ── Multi-line Orders (expected SAP behaviour) ─────────────────────────────────
order_counts      = df_combined.groupby('Order').size()
multi_line_orders = order_counts[order_counts > 1]
print(f"Multi-line Orders    : {len(multi_line_orders):,} orders with >1 material line (expected)")

# ── True Duplicates: identical rows entered twice in SAP ──────────────────────
# Negatives are valid SAP reversals — kept as-is. Only exact duplicate rows removed.
rows_before = len(df_combined)

df_true_duplicates = df_combined[
    df_combined.duplicated(
        subset=['Order', 'Material', 'SLoc', 'Date', 'Quantity'],
        keep=False
    )
].sort_values(['Order', 'Material'])

df_true_duplicates.to_csv("Logs/duplicate_rows.csv", index=False)
print(f"True Duplicate Rows  : {len(df_true_duplicates):,} rows → Logs/duplicate_rows.csv")

# ── Remove duplicates, keep first occurrence ───────────────────────────────────
df_combined = df_combined.drop_duplicates(
    subset=['Order', 'Material', 'SLoc', 'Date', 'Quantity'],
    keep='first'
).reset_index(drop=True)

# ── Final Summary ──────────────────────────────────────────────────────────────
print(f"\n{'='*45}")
print(f"Total rows (raw)     : {rows_before:,}")
print(f"Missing Orders       : {len(df_missing_orders):,}")
print(f"Missing SLoc/Temp    : {len(df_missing_sloc):,}")
print(f"Duplicates removed   : {rows_before - len(df_combined):,}")
print(f"df_combined (clean)  : {len(df_combined):,} rows")
print(f"{'='*45}")

display(df_combined.head(5))

Missing Orders       : 60 rows → Logs/missing_order.csv
Missing SLoc         : 0 rows → Logs/missing_sloc.csv
Merge Fan-out Orders : 0 (clean)
Multi-line Orders    : 6,024 orders with >1 material line (expected)
True Duplicate Rows  : 168 rows → Logs/duplicate_rows.csv

Total rows (raw)     : 64,895
Missing Orders       : 60
Missing SLoc/Temp    : 0
Duplicates removed   : 88
df_combined (clean)  : 64,807 rows


,Material,SLoc,Order,Quantity,Equipment Number,Technician Name,Tavg,Tmax,Tmin,RH,Delta_T,Date,Month,Year,Season,Location,Region
0,111,5043,68160889,-1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
1,111,5043,68160889,1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
2,100,5030,68165456,-1,11565095,Kamal Kant,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
3,111,5030,68165435,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
4,111,5030,68165114,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1


### Step 6 - Export Process

The processed dataset is exported into a single Excel workbook (`Combined_Master_Data.xlsx`) containing three sheets:

1. **Master Data** – Complete transaction-level dataset with all original and derived features.
2. **Monthly Pivot** – Material-wise monthly consumption matrix with Month-Year columns.
3. **Material Monthly Aggregation** – Monthly aggregated consumption along with average environmental parameters (Tavg, Tmax, Tmin, RH, and Delta_T).

This structure provides detailed records, time-series consumption data, and aggregated data for further analysis and forecasting.

In [18]:
# ── Step 6: Export to Excel — Three Representations ───────────────────────────

# ── Sheet 1: Combined Master Data (transaction level) ─────────────────────────
df_sheet1 = df_combined.copy()

# ── Sheet 2: Monthly Consumption Pivot (Material × Month-Year) ────────────────
# Rows = Material, Columns = Jan-20, Feb-20, ... (chronological)
df_pivot = df_combined.copy()
df_pivot['Month_Year'] = pd.to_datetime(df_pivot['Date']).dt.to_period('M')

df_sheet2 = (
    df_pivot
    .groupby(['Material', 'Month_Year'])['Quantity']
    .sum()
    .unstack('Month_Year')
    .fillna(0)
    .astype(int)
)
df_sheet2.columns = [str(c) for c in df_sheet2.columns]  # e.g. '2020-01'
df_sheet2 = df_sheet2.reset_index()

# ── Sheet 3: Material-wise Monthly Aggregation with Environmental Columns ──────
# Group by Material + Month + Year, sum Quantity, average the weather columns
df_sheet3 = (
    df_combined
    .groupby(['Material', 'Year', 'Month', 'Season', 'Region'], as_index=False, dropna=False)
    .agg(
        Quantity  = ('Quantity', 'sum'),
        Tavg      = ('Tavg',     'mean'),
        Tmax      = ('Tmax',     'mean'),
        Tmin      = ('Tmin',     'mean'),
        RH        = ('RH',       'mean'),
        Delta_T   = ('Delta_T',  'mean'),
    )
    .round(2)
    .sort_values(['Material', 'Year', 'Month'])
    .reset_index(drop=True)
)

# ── Write all three sheets to one Excel file ───────────────────────────────────
output_path = "Combined_Master_Data.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_sheet1.to_excel(writer, sheet_name='Master Data', index=False)
    df_sheet2.to_excel(writer, sheet_name='Monthly Pivot', index=False)
    df_sheet3.to_excel(writer, sheet_name='Material Monthly Agg', index=False)

print(f"Exported: {output_path}")
print(f"  Sheet 1 — Master Data          : {len(df_sheet1):,} rows")
print(f"  Sheet 2 — Monthly Pivot        : {len(df_sheet2):,} materials × {len(df_sheet2.columns)-1} months")
print(f"  Sheet 3 — Material Monthly Agg : {len(df_sheet3):,} rows")

Exported: Combined_Master_Data.xlsx
  Sheet 1 — Master Data          : 64,807 rows
  Sheet 2 — Monthly Pivot        : 12 materials × 76 months
  Sheet 3 — Material Monthly Agg : 2,816 rows


### Step 7 — Demand Pattern Classification (SBC Matrix)

Classifies each material as **Smooth / Erratic / Intermittent / Lumpy** based on
demand frequency (ADI) and variability (CV²), so the right forecasting model can be
applied to each group. SAP reversal months (negative net) are treated as zero demand
for classification only — the real net value is kept in the Material Monthly Agg sheet.

In [20]:
# ── Step 7: Demand Pattern Classification (SBC Matrix) ────────────────────────

# ── Full project month range (consistent ADI denominator for all materials) ───
full_month_range = pd.period_range(
    start=df_combined['Date'].min().to_period('M'),
    end=df_combined['Date'].max().to_period('M'),
    freq='M'
)
total_months = len(full_month_range)
print(f"Project window : {full_month_range[0]} to {full_month_range[-1]} ({total_months} months)")

# ── Net monthly quantity per Material (SAP reversals netted via sum) ──────────
df_combined['Month_Year'] = pd.to_datetime(df_combined['Date']).dt.to_period('M')

monthly_net = (
    df_combined
    .groupby(['Material', 'Month_Year'])['Quantity']
    .sum()
)

# ── Classification function ────────────────────────────────────────────────────
# NOTE: Quantity is negative for real consumption in this SAP dataset (not a
# reversal). Only a TRUE reversal pair nets to 0. So classification must use
# magnitude (abs), never clip negatives to 0.
def classify_demand(material_series_raw):
    s = material_series_raw.reindex(full_month_range, fill_value=0)

    non_zero = s[s != 0]
    if len(non_zero) == 0:
        return None, None, 'No Demand'

    magnitude = non_zero.abs()

    adi = total_months / len(non_zero)
    cv2 = (magnitude.std() / magnitude.mean()) ** 2 if magnitude.mean() != 0 else 0

    if adi < 1.32 and cv2 < 0.49:
        category = 'Smooth'
    elif adi < 1.32 and cv2 >= 0.49:
        category = 'Erratic'
    elif adi >= 1.32 and cv2 < 0.49:
        category = 'Intermittent'
    else:
        category = 'Lumpy'

    return round(adi, 3), round(cv2, 3), category

# ── Apply per Material ─────────────────────────────────────────────────────────
classification_rows = []
for material, grp in monthly_net.groupby('Material'):
    s = grp.droplevel('Material')
    adi, cv2, category = classify_demand(s)
    classification_rows.append({
        'Material': str(material),
        'ADI': adi,
        'CV2': cv2,
        'Category': category
    })

df_classification = pd.DataFrame(classification_rows)
df_classification['Material'] = df_classification['Material'].astype(str)

print("\nClassification summary:")
print(df_classification['Category'].value_counts())
print()
display(df_classification)

# ── Build Monthly Pivot sheet (one row per Material) ──────────────────────────
df_sheet2 = monthly_net.unstack('Month_Year')
df_sheet2 = df_sheet2.reindex(columns=full_month_range, fill_value=0).fillna(0).astype(int)
df_sheet2.columns = [c.strftime('%b-%y') for c in full_month_range]
df_sheet2 = df_sheet2.reset_index()
df_sheet2['Material'] = df_sheet2['Material'].astype(str)

# ── Merge classification onto Monthly Pivot — single source of truth ─────────
df_sheet2 = df_sheet2.merge(df_classification, on='Material', how='left')

month_cols = [c.strftime('%b-%y') for c in full_month_range]
df_sheet2 = df_sheet2[['Material', 'ADI', 'CV2', 'Category'] + month_cols]

print(f"\nMonthly Pivot with classification : {len(df_sheet2):,} materials")
display(df_sheet2.head())

# ── Material Monthly Agg sheet — NO classification columns (avoids repetition) ─
df_sheet3 = (
    df_combined
    .groupby(['Material', 'Year', 'Month', 'Season', 'Region'], as_index=False, dropna=False)
    .agg(
        Quantity=('Quantity', 'sum'),
        Tavg=('Tavg', 'mean'),
        Tmax=('Tmax', 'mean'),
        Tmin=('Tmin', 'mean'),
        RH=('RH', 'mean'),
        Delta_T=('Delta_T', 'mean')
    )
    .round(2)
    .sort_values(['Material', 'Year', 'Month'])
    .reset_index(drop=True)
)
df_sheet3['Material'] = df_sheet3['Material'].astype(str)

# ── Split into 4 dataframes for downstream modelling ───────────────────────────
category_map = df_classification.set_index('Material')['Category']

df_smooth = (
    df_sheet3[df_sheet3['Material'].map(category_map) == 'Smooth']
    .reset_index(drop=True)
)
df_erratic = (
    df_sheet3[df_sheet3['Material'].map(category_map) == 'Erratic']
    .reset_index(drop=True)
)
df_intermittent = (
    df_sheet3[df_sheet3['Material'].map(category_map) == 'Intermittent']
    .reset_index(drop=True)
)
df_lumpy = (
    df_sheet3[df_sheet3['Material'].map(category_map) == 'Lumpy']
    .reset_index(drop=True)
)

print(f"\n{'='*45}")
print(f"df_smooth        : {df_smooth['Material'].nunique():,} materials, {len(df_smooth):,} rows")
print(f"df_erratic       : {df_erratic['Material'].nunique():,} materials, {len(df_erratic):,} rows")
print(f"df_intermittent  : {df_intermittent['Material'].nunique():,} materials, {len(df_intermittent):,} rows")
print(f"df_lumpy         : {df_lumpy['Material'].nunique():,} materials, {len(df_lumpy):,} rows")
print(f"{'='*45}")

display(df_sheet3.head())

# ── Export Excel — plain, no colours/formatting, single-pass atomic write ─────
import os, tempfile

output_path = "Combined_Master_Data.xlsx"
tmp_fd, tmp_path = tempfile.mkstemp(suffix=".xlsx", dir=".")
os.close(tmp_fd)

with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
    df_sheet1.to_excel(writer, sheet_name="Master Data", index=False)
    df_sheet2.to_excel(writer, sheet_name="Monthly Pivot", index=False)
    df_sheet3.to_excel(writer, sheet_name="Material Monthly Agg", index=False)

os.replace(tmp_path, output_path)

print(f"\nExported: {output_path}")
print(f"  Sheet 1 — Master Data          : {len(df_sheet1):,} rows")
print(f"  Sheet 2 — Monthly Pivot        : {len(df_sheet2):,} rows (incl. ADI/CV2/Category)")
print(f"  Sheet 3 — Material Monthly Agg : {len(df_sheet3):,} rows (no classification columns)")

Project window : 2020-01 to 2026-04 (76 months)

Classification summary:
Category
Lumpy      7
Erratic    3
Smooth     2
Name: count, dtype: int64



,Material,ADI,CV2,Category
0,100,1.000,0.337,Smooth
1,101,1.000,0.584,Erratic
2,102,1.000,0.267,Smooth
3,103,1.000,0.740,Erratic
4,104,1.767,1.816,Lumpy
5,105,1.333,2.727,Lumpy
6,106,2.054,1.686,Lumpy
7,107,1.407,2.646,Lumpy
8,108,1.434,2.618,Lumpy
9,109,1.267,2.769,Erratic



Monthly Pivot with classification : 12 materials


,Material,ADI,CV2,Category,Jan-20,Feb-20,Mar-20,Apr-20,May-20,Jun-20,...,Jul-25,Aug-25,Sep-25,Oct-25,Nov-25,Dec-25,Jan-26,Feb-26,Mar-26,Apr-26
0,100,1.000,0.337,Smooth,-78,-74,-56,-16,-15,-100,...,-374,-336,-331,-286,-300,-277,-368,-467,-577,-427
1,101,1.000,0.584,Erratic,-4,-3,-2,-2,-12,-3,...,-83,-74,-85,-101,-97,-86,-84,-96,-111,-77
2,102,1.000,0.267,Smooth,-36,-36,-21,-6,-29,-37,...,-149,-188,-175,-194,-323,-221,-237,-218,-227,-241
3,103,1.000,0.740,Erratic,-36,-30,-32,-7,-24,-49,...,-208,-356,-210,-257,-415,-430,-296,-354,-650,-388
4,104,1.767,1.816,Lumpy,0,0,0,0,0,-1,...,-108,-72,-76,-88,-109,-100,-112,-95,-88,-103



df_smooth        : 2 materials, 863 rows
df_erratic       : 3 materials, 918 rows
df_intermittent  : 0 materials, 0 rows
df_lumpy         : 7 materials, 1,035 rows


,Material,Year,Month,Season,Region,Quantity,Tavg,Tmax,Tmin,RH,Delta_T
0,100,2020,1,Winter,East,-1,16.99,21.69,12.72,87.30,8.97
1,100,2020,1,Winter,North1,-37,12.70,20.19,6.80,66.76,13.40
2,100,2020,1,Winter,South,-11,23.63,30.83,17.78,62.98,13.05
3,100,2020,1,Winter,West1,-25,21.99,28.91,16.59,67.28,12.32
4,100,2020,1,Winter,West2,-4,20.16,27.60,13.83,64.22,13.77



Exported: Combined_Master_Data.xlsx
  Sheet 1 — Master Data          : 64,807 rows
  Sheet 2 — Monthly Pivot        : 12 rows (incl. ADI/CV2/Category)
  Sheet 3 — Material Monthly Agg : 2,816 rows (no classification columns)
